# Milestone 2: Data Cleaning (Polars) & Profiling (DuckDB)

Tahap ini bertujuan untuk membersihkan data mentah dari nilai yang kosong atau anomali menggunakan efisiensi **Polars**, lalu menyimpannya dalam format `.parquet` yang super cepat.
Setelah itu, kita akan menggunakan fitur `SUMMARIZE` dari mesin analitik **DuckDB** untuk mendapatkan ringkasan profil data secara instan.

In [3]:
import polars as pl
import duckdb
import glob
import os
import warnings
warnings.filterwarnings('ignore')

file_list = glob.glob("../data/raw/*.tsv.gz")
lazy_frames = []
for f in file_list:
    lf = pl.scan_csv(
        f, 
        separator='\t', 
        has_header=False,
        quote_char=None,
        ignore_errors=True,
        new_columns=["halaman_asal", "halaman_tujuan", "tipe_link", "jumlah_klik"]
    )
    lazy_frames.append(lf)

df_raw = pl.concat(lazy_frames)
print("Data mentah siap dibersihkan!")

Data mentah siap dibersihkan!


### 1. Data Cleaning menggunakan Polars
Sesuai standar kuliah, kita menerapkan aturan pembersihan (*data quality rules*) berikut:
1. Menghapus baris duplikat (Duplikasi).
2. Menghapus baris yang mengandung nilai kosong (Missing values).
3. Memastikan `jumlah_klik` selalu lebih dari 0 (Penanganan Outlier/Anomali).
4. Merapikan format teks: Halaman Wikipedia menggunakan garis bawah (`_`) untuk spasi. Kita akan menggantinya menjadi spasi biasa agar lebih mudah dibaca saat divisualisasikan nanti.

In [4]:
df_clean = (
    df_raw
    .unique()      
    .drop_nulls()  
    .filter(pl.col("jumlah_klik") > 0)  
    .with_columns([
        pl.col("halaman_asal").str.replace_all("_", " ").alias("halaman_asal"),
        pl.col("halaman_tujuan").str.replace_all("_", " ").alias("halaman_tujuan")
    ])
)

df_final = df_clean.collect()
print(f"Total baris setelah dibersihkan: {df_final.shape[0]:,}")

Total baris setelah dibersihkan: 2,396,531


### 2. Menyimpan Hasil ke Format Parquet
Data yang sudah bersih akan kita simpan ke dalam format `.parquet` di folder `data/processed/`. Format ini ratusan kali lebih efisien dan cepat dibaca oleh DuckDB dibandingkan file `.csv`.

In [5]:
os.makedirs("../data/processed", exist_ok=True)

PROCESSED_PATH = "../data/processed/clean_clickstream.parquet"
df_final.write_parquet(PROCESSED_PATH)
print(f"Data berhasil disimpan secara efisien ke: {PROCESSED_PATH}")

Data berhasil disimpan secara efisien ke: ../data/processed/clean_clickstream.parquet


### 3. Profiling Lanjutan menggunakan DuckDB `SUMMARIZE`
Di Milestone 1 kita profiling manual dengan Polars. Kini, kita mematuhi aturan untuk menggunakan DuckDB guna melakukan profil otomatis pada file Parquet kita menggunakan perintah sakti: `SUMMARIZE`.

In [6]:
query = f"SUMMARIZE SELECT * FROM '{PROCESSED_PATH}'"

duck_profile = duckdb.sql(query).pl()

print("Hasil Profiling Otomatis DuckDB SUMMARIZE:")
display(duck_profile)

Hasil Profiling Otomatis DuckDB SUMMARIZE:


column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""halaman_asal""","""VARCHAR""","""""Gadis \""Hawk Tuah\""""""","""＃2019GantiPresiden""",69804,null,null,null,null,null,2396531,0.00
"""halaman_tujuan""","""VARCHAR""","""!Kung""","""＃LoveSignal""",246787,null,null,null,null,null,2396531,0.00
"""tipe_link""","""VARCHAR""","""external""","""other""",2,null,null,null,null,null,2396531,0.00
"""jumlah_klik""","""BIGINT""","""10""","""3055641""",7631,"""100.06162115157284""","""2468.439211678672""","""14""","""23""","""53""",2396531,0.00
